# 01 - FER2013 数据探索与分析

目标：
1. 加载并理解FER2013数据集的结构
2. 分析类别分布，识别不平衡问题
3. 可视化各类别样本
4. 为后续训练决策（采样策略、loss加权）提供数据支撑

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import yaml

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# 加载配置
with open('../../configs/training_config.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

CLASS_NAMES = config['data']['class_names']
print(f"类别数量: {len(CLASS_NAMES)}")
print(f"类别名称: {CLASS_NAMES}")

## 1. 加载数据

In [ ]:
DATA_PATH = Path('../../data/fer2013.csv')

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"数据集未找到: {DATA_PATH}\n"
        "请从 https://www.kaggle.com/datasets/msambare/fer2013 下载并放入data/目录"
    )

df = pd.read_csv(DATA_PATH)
print(f"数据集形状: {df.shape}")
print(f"列名: {df.columns.tolist()}")
print(f"\n前5行:")
df.head()

## 2. 类别分布分析

In [ ]:
# 统计各类别样本数
emotion_counts = df['emotion'].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 柱状图
bars = axes[0].bar(range(len(CLASS_NAMES)), emotion_counts.values, color='steelblue')
axes[0].set_xticks(range(len(CLASS_NAMES)))
axes[0].set_xticklabels(CLASS_NAMES, rotation=45)
axes[0].set_ylabel('样本数')
axes[0].set_title('FER2013 类别分布')
for bar, count in zip(bars, emotion_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 50,
                 str(count), ha='center', va='bottom', fontsize=9)

# 饼图
axes[1].pie(emotion_counts.values, labels=CLASS_NAMES, autopct='%1.1f%%',
            startangle=90, colors=sns.color_palette('Set2', len(CLASS_NAMES)))
axes[1].set_title('FER2013 类别比例')

plt.tight_layout()
plt.savefig('../../analysis/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n--- 类别分布详情 ---")
for i, name in enumerate(CLASS_NAMES):
    count = emotion_counts.get(i, 0)
    pct = count / emotion_counts.sum() * 100
    print(f"{name:>10s}: {count:5d} ({pct:5.1f}%)")

# 计算不平衡比例
max_count = emotion_counts.max()
min_count = emotion_counts.min()
print(f"\n最大类/最小类比例: {max_count/min_count:.1f}:1")
print(f"结论: {'严重不平衡' if max_count/min_count > 5 else '轻微不平衡'}")

## 3. 样本可视化

In [ ]:
def parse_pixels(pixel_string):
    """将FER2013的像素字符串转为48x48 numpy数组"""
    pixels = np.array(pixel_string.split(), dtype=np.float32)
    return pixels.reshape(48, 48)

# 每个类别展示5个样本
fig, axes = plt.subplots(len(CLASS_NAMES), 5, figsize=(15, 3*len(CLASS_NAMES)))

for emotion_idx in range(len(CLASS_NAMES)):
    subset = df[df['emotion'] == emotion_idx].sample(5, random_state=42)
    for col, (_, row) in enumerate(subset.iterrows()):
        img = parse_pixels(row['pixels'])
        axes[emotion_idx, col].imshow(img, cmap='gray')
        axes[emotion_idx, col].axis('off')
        if col == 0:
            axes[emotion_idx, col].set_ylabel(CLASS_NAMES[emotion_idx], fontsize=12)

plt.suptitle('FER2013 各类别样本示例', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../../analysis/sample_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. 数据集划分与Usage字段分析

In [ ]:
# FER2013的Usage字段：Training / PublicTest / PrivateTest
if 'Usage' in df.columns:
    usage_counts = df['Usage'].value_counts()
    print("Usage分布:")
    print(usage_counts)
    print(f"\n训练集: {usage_counts.get('Training', 0)}")
    print(f"验证集: {usage_counts.get('PublicTest', 0)}")
    print(f"测试集: {usage_counts.get('PrivateTest', 0)}")
else:
    print("数据集无Usage字段，需手动划分")

## 5. 类别权重计算（用于加权CrossEntropyLoss）

In [ ]:
# 逆频率加权
total = emotion_counts.sum()
class_weights = total / (len(CLASS_NAMES) * emotion_counts.sort_index().values)

print("建议的类别权重 (inverse frequency):")
for i, (name, weight) in enumerate(zip(CLASS_NAMES, class_weights)):
    print(f"  {name:>10s}: {weight:.4f}")

import torch
weights_tensor = torch.FloatTensor(class_weights)
print(f"\nPyTorch Tensor: {weights_tensor}")
print("\n使用方式: nn.CrossEntropyLoss(weight=weights_tensor.to(device))")

## 6. 结论与决策建议

根据以上分析，记录以下决策点：
- [ ] 类别不平衡策略选择：weighted_loss / oversampling / 无处理
- [ ] Disgust类是否需要特殊处理（样本极少）
- [ ] 数据增强策略确认
- [ ] 训练/验证/测试划分方案确认